In [2]:
# ==========================================================
# Step 1: Install & Import Required Libraries
# ==========================================================

!pip -q install catboost

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from scipy.stats import randint, uniform

from catboost import CatBoostRegressor

import joblib

print("=" * 70)
print("All Libraries Imported Successfully")
print("=" * 70)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.9 MB/s eta 0:00:00
All Libraries Imported Successfully


In [3]:
# ==========================================================
# Step 2 : Upload Dataset
# ==========================================================

from google.colab import files

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

print("=" * 70)
print("Dataset Uploaded Successfully")
print("=" * 70)

print("Uploaded File :", file_name)

Saving Weather3 (2).csv to Weather3 (2).csv
Dataset Uploaded Successfully
Uploaded File : Weather3 (2).csv


In [4]:
# ==========================================================
# Step 3 : Read Dataset
# ==========================================================

df = pd.read_csv(file_name)

# Convert datetime column
df["datetime"] = pd.to_datetime(df["datetime"])

# Sort by time
df = df.sort_values("datetime").reset_index(drop=True)

print("=" * 70)
print("Dataset Loaded Successfully")
print("=" * 70)

print("\nShape :", df.shape)

print("\nColumns :")
print(df.columns.tolist())

print("\nData Types")
display(df.dtypes)

print("\nFirst 5 Rows")
display(df.head())

print("\nLast 5 Rows")
display(df.tail())

Dataset Loaded Successfully

Shape : (298690, 4)

Columns :
['_id', 'datetime', 'temp', 'rh']

Data Types


,0
_id,object
datetime,datetime64[ns]
temp,float64
rh,float64



First 5 Rows


,_id,datetime,temp,rh
0,65ec7952a00f951ad6604b13,2017-12-31 18:30:00,19.12,68.06
1,65ec7952a00f951ad6604b14,2017-12-31 18:45:00,19.06,68.14
2,65ec7952a00f951ad6604b15,2017-12-31 19:00:00,18.96,68.32
3,65ec7952a00f951ad6604b16,2017-12-31 19:15:00,18.97,66.54
4,65ec7952a00f951ad6604b17,2017-12-31 19:30:00,18.90,67.32



Last 5 Rows


,_id,datetime,temp,rh
298685,6a4f0d562d885f82fd824b6d,2026-07-09 01:45:00,28.926667,84.660000
298686,6a4f0d562d885f82fd824b6e,2026-07-09 02:00:00,28.580000,86.646667
298687,6a4f0d562d885f82fd824b6f,2026-07-09 02:15:00,28.666667,86.153333
298688,6a4f0d562d885f82fd824b70,2026-07-09 02:30:00,29.080000,84.446667
298689,6a4f0d562d885f82fd824b71,2026-07-09 02:45:00,29.344444,83.577778


In [5]:
# ==========================================================
# Step 4 : Data Cleaning
# ==========================================================

# Remove duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

# Remove rows with missing values
df = df.dropna().reset_index(drop=True)

# Remove impossible Relative Humidity values
df = df[(df["rh"] >= 0) & (df["rh"] <= 100)]

# Sort data chronologically
df = df.sort_values("datetime").reset_index(drop=True)

print("=" * 70)
print("Data Cleaning Completed Successfully")
print("=" * 70)

print(f"Total Records : {len(df):,}")

print("\nMissing Values")
display(df.isnull().sum())

print("\nDataset Shape")
print(df.shape)

print("\nFirst 5 Rows")
display(df.head())

Data Cleaning Completed Successfully
Total Records : 298,575

Missing Values


,0
_id,0
datetime,0
temp,0
rh,0



Dataset Shape
(298575, 4)

First 5 Rows


,_id,datetime,temp,rh
0,65ec7952a00f951ad6604b13,2017-12-31 18:30:00,19.12,68.06
1,65ec7952a00f951ad6604b14,2017-12-31 18:45:00,19.06,68.14
2,65ec7952a00f951ad6604b15,2017-12-31 19:00:00,18.96,68.32
3,65ec7952a00f951ad6604b16,2017-12-31 19:15:00,18.97,66.54
4,65ec7952a00f951ad6604b17,2017-12-31 19:30:00,18.90,67.32


In [6]:
# ==========================================================
# Step 5 : Create Seasons
# ==========================================================

def get_season(month):

    if month in [12, 1, 2]:
        return "Winter"

    elif month in [3, 4, 5]:
        return "Summer"

    elif month in [6, 7, 8, 9]:
        return "Monsoon"

    else:
        return "Post-Monsoon"


# Create Season Column
df["season"] = df["datetime"].dt.month.apply(get_season)

print("=" * 70)
print("Season Column Created Successfully")
print("=" * 70)

print("\nSeason-wise Record Count")
display(df["season"].value_counts())

print("\nFirst 10 Rows")
display(df[["datetime", "temp", "rh", "season"]].head(10))

Season Column Created Successfully

Season-wise Record Count


,count
season,
Monsoon,97284
Summer,79488
Winter,74998
Post-Monsoon,46805



First 10 Rows


,datetime,temp,rh,season
0,2017-12-31 18:30:00,19.12,68.06,Winter
1,2017-12-31 18:45:00,19.06,68.14,Winter
2,2017-12-31 19:00:00,18.96,68.32,Winter
3,2017-12-31 19:15:00,18.97,66.54,Winter
4,2017-12-31 19:30:00,18.90,67.32,Winter
5,2017-12-31 19:45:00,18.63,69.68,Winter
6,2017-12-31 20:00:00,18.48,70.23,Winter
7,2017-12-31 20:15:00,18.28,71.75,Winter
8,2017-12-31 20:30:00,18.24,74.54,Winter
9,2017-12-31 20:45:00,18.08,77.97,Winter


In [7]:
# ==========================================================
# Step 6 : Create Season-wise Datasets
# ==========================================================

winter_df = df[df["season"] == "Winter"].copy().reset_index(drop=True)

summer_df = df[df["season"] == "Summer"].copy().reset_index(drop=True)

monsoon_df = df[df["season"] == "Monsoon"].copy().reset_index(drop=True)

post_df = df[df["season"] == "Post-Monsoon"].copy().reset_index(drop=True)

print("="*70)
print("Season-wise Datasets Created Successfully")
print("="*70)

print(f"Winter         : {winter_df.shape}")
print(f"Summer         : {summer_df.shape}")
print(f"Monsoon        : {monsoon_df.shape}")
print(f"Post-Monsoon   : {post_df.shape}")

Season-wise Datasets Created Successfully
Winter         : (74998, 5)
Summer         : (79488, 5)
Monsoon        : (97284, 5)
Post-Monsoon   : (46805, 5)


In [8]:
# ==========================================================
# Step 7 : Advanced Feature Engineering
# ==========================================================

def create_features(df):

    # -----------------------------
    # Time Features
    # -----------------------------
    df["hour"] = df["datetime"].dt.hour
    df["day"] = df["datetime"].dt.day
    df["month"] = df["datetime"].dt.month
    df["dayofweek"] = df["datetime"].dt.dayofweek

    # -----------------------------
    # Cyclic Features
    # -----------------------------
    df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)

    df["month_sin"] = np.sin(2*np.pi*df["month"]/12)
    df["month_cos"] = np.cos(2*np.pi*df["month"]/12)

    # -----------------------------
    # Temperature Lags
    # -----------------------------
    for lag in [1,2,3,6,12,24]:
        df[f"temp_lag_{lag}"] = df["temp"].shift(lag)

    # -----------------------------
    # RH Lags
    # -----------------------------
    for lag in [1,2,3,6,12,24]:
        df[f"rh_lag_{lag}"] = df["rh"].shift(lag)

    # -----------------------------
    # Rolling Statistics
    # -----------------------------
    df["temp_roll_mean_3"] = df["temp"].rolling(3).mean()
    df["temp_roll_mean_6"] = df["temp"].rolling(6).mean()
    df["temp_roll_mean_12"] = df["temp"].rolling(12).mean()

    df["temp_roll_std_3"] = df["temp"].rolling(3).std()
    df["temp_roll_std_6"] = df["temp"].rolling(6).std()
    df["temp_roll_std_12"] = df["temp"].rolling(12).std()

    df["rh_roll_mean_3"] = df["rh"].rolling(3).mean()
    df["rh_roll_mean_6"] = df["rh"].rolling(6).mean()
    df["rh_roll_mean_12"] = df["rh"].rolling(12).mean()

    df["rh_roll_std_3"] = df["rh"].rolling(3).std()
    df["rh_roll_std_6"] = df["rh"].rolling(6).std()
    df["rh_roll_std_12"] = df["rh"].rolling(12).std()

    # -----------------------------
    # Trend Features
    # -----------------------------
    df["temp_change"] = df["temp"].diff()

    df["rh_change"] = df["rh"].diff()

    # -----------------------------
    # Remove NaN rows
    # -----------------------------
    df = df.dropna().reset_index(drop=True)

    return df


winter_df = create_features(winter_df)

summer_df = create_features(summer_df)

monsoon_df = create_features(monsoon_df)

post_df = create_features(post_df)

print("="*70)
print("Advanced Feature Engineering Completed")
print("="*70)

print(f"Winter         : {winter_df.shape}")
print(f"Summer         : {summer_df.shape}")
print(f"Monsoon        : {monsoon_df.shape}")
print(f"Post-Monsoon   : {post_df.shape}")

Advanced Feature Engineering Completed
Winter         : (74974, 39)
Summer         : (79464, 39)
Monsoon        : (97260, 39)
Post-Monsoon   : (46781, 39)


In [9]:
# ==========================================================
# Step 8 : Prepare Features & Target
# ==========================================================

feature_columns = [

    # Time Features
    "hour",
    "day",
    "month",
    "dayofweek",

    # Cyclic Features
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",

    # Temperature Lags
    "temp_lag_1",
    "temp_lag_2",
    "temp_lag_3",
    "temp_lag_6",
    "temp_lag_12",
    "temp_lag_24",

    # RH Lags
    "rh_lag_1",
    "rh_lag_2",
    "rh_lag_3",
    "rh_lag_6",
    "rh_lag_12",
    "rh_lag_24",

    # Temperature Rolling
    "temp_roll_mean_3",
    "temp_roll_mean_6",
    "temp_roll_mean_12",

    "temp_roll_std_3",
    "temp_roll_std_6",
    "temp_roll_std_12",

    # RH Rolling
    "rh_roll_mean_3",
    "rh_roll_mean_6",
    "rh_roll_mean_12",

    "rh_roll_std_3",
    "rh_roll_std_6",
    "rh_roll_std_12",

    # Trend
    "temp_change",
    "rh_change"

]

target = "rh"

X_winter = winter_df[feature_columns]
y_winter = winter_df[target]

X_summer = summer_df[feature_columns]
y_summer = summer_df[target]

X_monsoon = monsoon_df[feature_columns]
y_monsoon = monsoon_df[target]

X_post = post_df[feature_columns]
y_post = post_df[target]

print("="*70)
print("Feature Preparation Completed")
print("="*70)

print(f"Winter         : {X_winter.shape}")
print(f"Summer         : {X_summer.shape}")
print(f"Monsoon        : {X_monsoon.shape}")
print(f"Post-Monsoon   : {X_post.shape}")

Feature Preparation Completed
Winter         : (74974, 34)
Summer         : (79464, 34)
Monsoon        : (97260, 34)
Post-Monsoon   : (46781, 34)


In [10]:
# ==========================================================
# Step 9 : Train-Test Split
# ==========================================================

from sklearn.model_selection import train_test_split

X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_winter, y_winter,
    test_size=0.20,
    shuffle=False
)

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_summer, y_summer,
    test_size=0.20,
    shuffle=False
)

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_monsoon, y_monsoon,
    test_size=0.20,
    shuffle=False
)

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_post, y_post,
    test_size=0.20,
    shuffle=False
)

print("="*70)
print("Train-Test Split Completed")
print("="*70)

print(f"Winter Train : {X_train_w.shape} | Test : {X_test_w.shape}")
print(f"Summer Train : {X_train_s.shape} | Test : {X_test_s.shape}")
print(f"Monsoon Train: {X_train_m.shape} | Test : {X_test_m.shape}")
print(f"Post Train   : {X_train_p.shape} | Test : {X_test_p.shape}")

Train-Test Split Completed
Winter Train : (59979, 34) | Test : (14995, 34)
Summer Train : (63571, 34) | Test : (15893, 34)
Monsoon Train: (77808, 34) | Test : (19452, 34)
Post Train   : (37424, 34) | Test : (9357, 34)


In [3]:
# ==========================================================
# Step 10 : Optuna Optimization (Monsoon Only)
# ==========================================================

!pip -q install optuna

import optuna
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score

# ----------------------------------------------------------
# Split Monsoon Train into Train & Validation
# ----------------------------------------------------------

split = int(len(X_train_m) * 0.9)

X_tr = X_train_m.iloc[:split]
y_tr = y_train_m.iloc[:split]

X_val = X_train_m.iloc[split:]
y_val = y_train_m.iloc[split:]

# ----------------------------------------------------------
# Objective Function
# ----------------------------------------------------------

def objective(trial):

    params = {

        "iterations": trial.suggest_int("iterations", 2000, 10000),

        "depth": trial.suggest_int("depth", 4, 10),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.005,
            0.05,
            log=True
        ),

        "l2_leaf_reg": trial.suggest_int(
            "l2_leaf_reg",
            1,
            20
        ),

        "bagging_temperature": trial.suggest_float(
            "bagging_temperature",
            0,
            5
        ),

        "random_strength": trial.suggest_float(
            "random_strength",
            0,
            5
        ),

        "border_count": trial.suggest_categorical(
            "border_count",
            [64,128,254]
        ),

        "loss_function":"RMSE",

        "eval_metric":"RMSE",

        "random_seed":42,

        "verbose":0

    }

    model = CatBoostRegressor(**params)

    model.fit(

        X_tr,
        y_tr,

        eval_set=(X_val,y_val),

        use_best_model=True,

        early_stopping_rounds=300

    )

    pred = model.predict(X_val)

    return r2_score(y_val,pred)

# ----------------------------------------------------------
# Run Optuna
# ----------------------------------------------------------

study = optuna.create_study(direction="maximize")

study.optimize(

    objective,

    n_trials=150,

    show_progress_bar=True

)

print("="*70)
print("Best Parameters")
print(study.best_params)

print("\nBest Validation R²")
print(study.best_value)

print("="*70)

# ----------------------------------------------------------
# Train Final Monsoon Model
# ----------------------------------------------------------

monsoon_model = CatBoostRegressor(

    **study.best_params,

    loss_function="RMSE",

    eval_metric="RMSE",

    random_seed=42,

    verbose=200

)

monsoon_model.fit(

    X_train_m,

    y_train_m,

    eval_set=(X_test_m,y_test_m),

    use_best_model=True,

    early_stopping_rounds=300

)

print("="*70)
print("Optimized Monsoon Model Trained Successfully")
print("="*70)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 16.1 MB/s eta 0:00:00


ModuleNotFoundError: No module named 'catboost'

In [ ]:
# ==========================================================
# Step 11 : Generate Predictions & Evaluate Model
# ==========================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ----------------------------------------------------------
# Function
# ----------------------------------------------------------

def evaluate_model(model, X_test, y_test, season):

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)

    rmse = np.sqrt(mean_squared_error(y_test, predictions))

    r2 = r2_score(y_test, predictions)

    return [season, mae, rmse, r2], predictions


# ---------------- Winter ----------------

winter_result, winter_predictions = evaluate_model(
    winter_model,
    X_test_w,
    y_test_w,
    "Winter"
)

# ---------------- Summer ----------------

summer_result, summer_predictions = evaluate_model(
    summer_model,
    X_test_s,
    y_test_s,
    "Summer"
)

# ---------------- Monsoon ----------------

monsoon_result, monsoon_predictions = evaluate_model(
    monsoon_model,
    X_test_m,
    y_test_m,
    "Monsoon"
)

# ---------------- Post-Monsoon ----------------

post_result, post_predictions = evaluate_model(
    post_model,
    X_test_p,
    y_test_p,
    "Post-Monsoon"
)

# ==========================================================
# Performance Table
# ==========================================================

results = pd.DataFrame(

    [
        winter_result,
        summer_result,
        monsoon_result,
        post_result
    ],

    columns=[
        "Season",
        "MAE",
        "RMSE",
        "R² Score"
    ]

)

print("="*70)
print("Season-wise CatBoost Model Performance")
print("="*70)

display(results)

In [2]:
# ==========================================================
# Step 12 : Save Models & Prediction Results
# ==========================================================

import joblib

# ----------------------------------------------------------
# Save Models
# ----------------------------------------------------------

joblib.dump(winter_model, "winter_catboost.pkl")
joblib.dump(summer_model, "summer_catboost.pkl")
joblib.dump(monsoon_model, "monsoon_catboost.pkl")
joblib.dump(post_model, "post_monsoon_catboost.pkl")

print("✅ Models Saved Successfully")


# ----------------------------------------------------------
# Create Prediction DataFrames
# ----------------------------------------------------------

winter_pred_df = pd.DataFrame({
    "Actual_RH": y_test_w.values,
    "Predicted_RH": winter_predictions
})

summer_pred_df = pd.DataFrame({
    "Actual_RH": y_test_s.values,
    "Predicted_RH": summer_predictions
})

monsoon_pred_df = pd.DataFrame({
    "Actual_RH": y_test_m.values,
    "Predicted_RH": monsoon_predictions
})

post_pred_df = pd.DataFrame({
    "Actual_RH": y_test_p.values,
    "Predicted_RH": post_predictions
})


# ----------------------------------------------------------
# Save Prediction Excel
# ----------------------------------------------------------

with pd.ExcelWriter("Seasonal_RH_Predictions.xlsx") as writer:

    winter_pred_df.to_excel(writer,
                            sheet_name="Winter",
                            index=False)

    summer_pred_df.to_excel(writer,
                            sheet_name="Summer",
                            index=False)

    monsoon_pred_df.to_excel(writer,
                             sheet_name="Monsoon",
                             index=False)

    post_pred_df.to_excel(writer,
                          sheet_name="Post-Monsoon",
                          index=False)

print("✅ Prediction Excel Saved")


# ----------------------------------------------------------
# Save Performance Table
# ----------------------------------------------------------

results.to_excel(
    "Seasonal_RH_Model_Performance.xlsx",
    index=False
)

print("✅ Performance Report Saved")

print("="*70)
print("All Files Saved Successfully")
print("="*70)

NameError: name 'winter_model' is not defined